In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

# ※ Quiz : 경주여행과 전주여행에 대한 최빈단어 시각화와 유사도 분석
- (1) naver open API를 활용하여 블로그에 "경주여행","전주여행"을 각각 500건씩검색하여 백업                 (data/quiz/naver.csv)
    * 백업 파일 내용(query, no, title, link, description, total, text(title + ' '+description)
- (2) naver.csv에서 total_text를 품사태깅(naver_pos.csv)
    * 파일 내용 : query, no, token, pos
- (3) 명사만 추출(naver_pos_nouns.csv)
    * query, token, pos
- (4) 빈도분석 백업(naver_pos_nouns_count.csv)
    * token, 경주빈도, 전주빈도, 빈도합
- (5) 빈도시각화(워드클라우드, Text.plot)
    * 이미지 저장

In [2]:
# . env가져오기
from dotenv import load_dotenv
import os
load_dotenv()

True

In [10]:
# 네이버 개발자 센터에 있는 소스
import os
import sys
import urllib.request
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")
encText = urllib.parse.quote("경주 여행")
url = "https://openapi.naver.com/v1/search/blog?query=" + encText # JSON 결과
# url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8')[:200])
else:
    print("Error Code:" + rescode)

{
	"lastBuildDate":"Thu, 03 Sep 2026 17:24:25 +0900",
	"total":2735078,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"3월의 <b>경주여행<\/b>.",
			"link":"https:\/\/lje77777.tistory.com\/7132",
			"


In [17]:
# 문자 -> dict
import json
from html import unescape # description에 있는 &lt;(특수문자)를 <로 변경
import requests
import pandas as pd
import re # 특수문자 제거

In [19]:
query = "경주 여행"
start = 1
# url =f'https://openapi.naver.com/v1/search/blog.json?query={query}&display=100&start={start}'

url = "https://openapi.naver.com/v1/search/blog.json"
params = {
    'query':query,
    'display':100,
    'start':start
}
headers ={
    "X-Naver-Client-Id":client_id,
    "X-Naver-Client-Secret":client_secret
}

response = requests.get(url,headers=headers, params=params)
# 문자 -> dict
items = json.loads(response.text)['items']
items = response.json()['items']
items[:2]

[{'title': '<b>경주여행</b>, 계림 <b>경주</b>역사유적지구 + 황간 대표맛집 유니짜장 덕승관',
  'link': 'https://minui.tistory.com/178',
  'description': '<b>경주여행</b>, 계림<b>경주</b>역사유적지구 + 황간 대표맛집 유니짜장 덕승관 <b>경주 여행</b>에서 빼놓을 수 없는 코스 첨성대가 있는 계림 <b>경주</b>역사유적지구입니다. 한번쯤 수학<b>여행</b>으로도 와본 곳이기도하죠^^ 전 날... ',
  'bloggername': '미루의 공감라이프',
  'bloggerlink': 'https://minui.tistory.com/',
  'postdate': '20190104'},
 {'title': '[<b>경주여행</b>] 교리김밥 봉황대점 - 김밥2줄 9,000원. 쎈데?!ㅋ',
  'link': 'https://jisuni-1116.tistory.com/326',
  'description': '<b>경주여행</b> 먹킷리스트 중에 하나였던 교리김밥 <b>경주</b> 황리단길 끝자락? 무튼 핫플레이스 거리에서 조금만 나가면 교리김밥 봉황대점이있더라. 그 유명한 교리김밥을 먹으로 고고고. 토욜 아점쯤이었는데... ',
  'bloggername': '★먹고 놀자★',
  'bloggerlink': 'https://jisuni-1116.tistory.com/',
  'postdate': '20210429'}]

In [ ]:
# title의 <b>없애기
item = items[4]
title = 